# Wheel Data Acquisition: From NIDQ Rotary Encoder to Processed Position

This notebook demonstrates the relationship between:
1. **Raw NIDQ rotary encoder signals** - TTL pulses from the quadrature encoder
2. **Processed wheel data** - Position, velocity, and detected movements

The wheel position is measured using a quadrature rotary encoder that outputs two phase-shifted digital signals (channels A and B). These signals are recorded by the NIDQ system and decoded to compute the wheel position.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

# Enable matplotlib inline
%matplotlib inline

## Load NWB Files from DANDI

We load both the raw NWB file (containing NIDQ data) and the processed NWB file (containing wheel position/velocity).

In [ ]:
# Connect to DANDI and get the dandiset
dandiset_id = "000409"
client = DandiAPIClient()
dandiset = client.get_dandiset(dandiset_id, "draft")

# Complete session with 2 probes - Session 1 (NYU-39, 2021-05-10, angelakilab)
session_eid = "6ed57216-498d-48a6-b48b-a243a34710ea"

# Fetch assets by EID
session_assets = [asset for asset in dandiset.get_assets() if session_eid in asset.path]
raw_asset = next((asset for asset in session_assets if "desc-raw" in asset.path), None)
processed_asset = next((asset for asset in session_assets if "desc-processed" in asset.path), None)

print(f"Session EID: {session_eid}")
print(f"\nRaw file:       {raw_asset.path if raw_asset else 'Not found'}")
print(f"Processed file: {processed_asset.path if processed_asset else 'Not found'}")

In [ ]:
# Load NWB files via streaming
raw_url = raw_asset.get_content_url(follow_redirects=1, strip_query=True)
processed_url = processed_asset.get_content_url(follow_redirects=1, strip_query=True)

# Wrap remfile with h5py.File for pynwb compatibility
raw_h5 = h5py.File(remfile.File(raw_url), mode="r")
processed_h5 = h5py.File(remfile.File(processed_url), mode="r")

io_raw = NWBHDF5IO(file=raw_h5, mode="r", load_namespaces=True)
io_processed = NWBHDF5IO(file=processed_h5, mode="r", load_namespaces=True)

nwbfile_raw = io_raw.read()
nwbfile_processed = io_processed.read()

print(f"Raw NWB loaded: {nwbfile_raw.identifier}")
print(f"Processed NWB loaded: {nwbfile_processed.identifier}")

## NIDQ Rotary Encoder Signals

The NIDQ (National Instruments DAQ) system records the raw TTL pulses from the rotary encoder. The encoder uses **quadrature encoding** with two channels (A and B) that are 90 degrees out of phase.

```
                                    NIDQ (Master Clock)        
                                    +------------------+       
Rotary Encoder --Phase A--TTL------>| P0.5 (channel 5) |
               --Phase B--TTL------>| P0.6 (channel 6) |
                                    +------------------+       
                                          @ ~1 kHz              
                                     (session time)             
```

Let's explore what NIDQ data is available in the raw NWB file.

In [ ]:
# Check what's in the acquisition module
print("Acquisition objects in raw NWB file:")
for name, obj in nwbfile_raw.acquisition.items():
    print(f"  {name}: {type(obj).__name__}")

# Check for NIDQ-related events
print("\nEvents in raw NWB file:")
if hasattr(nwbfile_raw, 'acquisition'):
    for name, obj in nwbfile_raw.acquisition.items():
        if 'rotary' in name.lower() or 'encoder' in name.lower() or 'nidq' in name.lower():
            print(f"  Found NIDQ-related: {name}")

In [ ]:
# Check for NIDQ events in the acquisition module
# NIDQ rotary encoder data is stored as LabeledEvents (TTL transitions)
nidq_events = {}
for name, obj in nwbfile_raw.acquisition.items():
    # Look for rotary encoder events (case-insensitive)
    if 'rotaryencoder' in name.lower().replace('_', ''):
        nidq_events[name] = obj
        print(f"Found: {name}")
        print(f"  Type: {type(obj).__name__}")
        if hasattr(obj, 'timestamps'):
            ts = obj.timestamps[:]
            print(f"  Number of events: {len(ts)}")
            print(f"  Time range: {ts[0]:.3f}s - {ts[-1]:.3f}s")
        print()

if not nidq_events:
    print("No rotary encoder events found in acquisition.")
    print("\\nNote: NIDQ data may not be available for all sessions.")
    print("The wheel position in processed files is derived from NIDQ during preprocessing.")

In [ ]:
# Visualize raw NIDQ rotary encoder events if available
if nidq_events:
    # Get both encoder channels
    encoder0 = nidq_events.get('EventsRotaryEncoder0')
    encoder1 = nidq_events.get('EventsRotaryEncoder1')
    
    if encoder0 and encoder1:
        # Get timestamps and labels for a short window
        ts0 = encoder0.timestamps[:]
        ts1 = encoder1.timestamps[:]
        labels0 = encoder0.data[:]  # 0 = phase_low, 1 = phase_high
        labels1 = encoder1.data[:]
        
        # Find a region with activity (first 1000 events)
        n_events = min(1000, len(ts0), len(ts1))
        t_start = min(ts0[0], ts1[0])
        t_end = max(ts0[n_events-1], ts1[n_events-1])
        
        fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
        
        # Plot Channel A (encoder0) events
        mask0 = ts0 <= t_end
        axes[0].vlines(ts0[mask0], 0, 1, colors=['blue' if label == 1 else 'lightblue' for label in labels0[mask0]], linewidth=0.5)
        axes[0].set_ylabel('Channel A', fontsize=10)
        axes[0].set_ylim(-0.1, 1.1)
        axes[0].set_title(f'Raw NIDQ Rotary Encoder Events ({t_start:.3f}s - {t_end:.3f}s)', fontsize=12, fontweight='bold')
        
        # Plot Channel B (encoder1) events  
        mask1 = ts1 <= t_end
        axes[1].vlines(ts1[mask1], 0, 1, colors=['red' if label == 1 else 'lightcoral' for label in labels1[mask1]], linewidth=0.5)
        axes[1].set_ylabel('Channel B', fontsize=10)
        axes[1].set_ylim(-0.1, 1.1)
        
        # Show event density (events per 10ms bin)
        bins = np.arange(t_start, t_end, 0.01)
        combined_events = np.concatenate([ts0[mask0], ts1[mask1]])
        axes[2].hist(combined_events, bins=bins, color='green', alpha=0.7)
        axes[2].set_ylabel('Events/10ms', fontsize=10)
        axes[2].set_xlabel('Time (s)', fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
        print("\\nRaw encoder event statistics:")
        print(f"  Channel A events: {len(ts0):,}")
        print(f"  Channel B events: {len(ts1):,}")
        print(f"  Total events: {len(ts0) + len(ts1):,}")
        print("  These TTL transitions are decoded to compute wheel position")

In [ ]:
# Compare raw NIDQ events with processed wheel position
# This shows how TTL pulses become position data
if nidq_events:
    encoder0 = nidq_events.get('EventsRotaryEncoder0')
    encoder1 = nidq_events.get('EventsRotaryEncoder1')
    
    if encoder0 and encoder1:
        # Load processed wheel position for comparison
        wheel_module = nwbfile_processed.processing["wheel"]
        wheel_pos = wheel_module["WheelPosition"]
        pos_data = wheel_pos.get_data_in_units()
        pos_times = wheel_pos.timestamps[:]
        
        # Find a short window with movement (around 10-15 seconds)
        t_window_start = 10.0
        t_window_end = 15.0
        
        # Get encoder events in this window
        ts0 = encoder0.timestamps[:]
        ts1 = encoder1.timestamps[:]
        mask0 = (ts0 >= t_window_start) & (ts0 <= t_window_end)
        mask1 = (ts1 >= t_window_start) & (ts1 <= t_window_end)
        
        # Get position data in this window
        pos_mask = (pos_times >= t_window_start) & (pos_times <= t_window_end)
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        
        # Top: Raw encoder events
        axes[0].eventplot([ts0[mask0], ts1[mask1]], colors=['blue', 'red'], 
                         lineoffsets=[0.7, 0.3], linelengths=0.4, linewidths=0.8)
        axes[0].set_ylabel('NIDQ Events', fontsize=10)
        axes[0].set_yticks([0.3, 0.7])
        axes[0].set_yticklabels(['Ch B', 'Ch A'])
        axes[0].set_title('Raw NIDQ Encoder Events vs Processed Wheel Position', fontsize=12, fontweight='bold')
        axes[0].set_ylim(0, 1)
        
        # Bottom: Processed wheel position
        axes[1].plot(pos_times[pos_mask], pos_data[pos_mask], 'g-', linewidth=1.5)
        axes[1].set_ylabel(f'Position ({wheel_pos.unit})', fontsize=10)
        axes[1].set_xlabel('Time (s)', fontsize=10)
        axes[1].grid(True, alpha=0.3)
        
        # Add annotation
        n_events = mask0.sum() + mask1.sum()
        axes[0].text(0.02, 0.95, f'{n_events} TTL events in window', transform=axes[0].transAxes,
                    fontsize=9, verticalalignment='top', 
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nIn this {t_window_end - t_window_start:.0f}s window:")
        print(f"  Channel A events: {mask0.sum()}")
        print(f"  Channel B events: {mask1.sum()}")
        print(f"  Position samples: {pos_mask.sum()}")
        print(f"  Position change: {pos_data[pos_mask][-1] - pos_data[pos_mask][0]:.3f} rad")

## Processed Wheel Data

The processed NWB file contains the wheel data that was computed from the NIDQ rotary encoder signals:

- **WheelPosition**: Absolute wheel angle in radians (raw, irregular timestamps)
- **WheelPositionSmoothed**: Position resampled to uniform 1000 Hz and smoothed
- **WheelSmoothedVelocity**: Angular velocity in rad/s (1000 Hz), derived from smoothed position
- **WheelSmoothedAcceleration**: Angular acceleration in rad/s^2 (1000 Hz), derived from smoothed position
- **WheelMovement**: Detected movement epochs

In [ ]:
# Access wheel processing module
wheel_module = nwbfile_processed.processing["wheel"]
print("Wheel processing module contents:")
for name in wheel_module.data_interfaces:
    print(f"  {name}")

In [ ]:
# Load wheel position
wheel_position = wheel_module["WheelPosition"]
position_data = wheel_position.get_data_in_units()
position_times = wheel_position.timestamps[:]

print("Wheel position:")
print(f"  Number of samples: {len(position_data)}")
print(f"  Time range: {position_times[0]:.3f}s - {position_times[-1]:.3f}s")
print(f"  Position range: {position_data.min():.2f} to {position_data.max():.2f} radians")
print(f"  Unit: {wheel_position.unit}")
print(f"  Reference frame: {wheel_position.reference_frame}")

In [ ]:
# Load wheel velocity (regularly sampled)
wheel_velocity = wheel_module["WheelSmoothedVelocity"]
velocity_data = wheel_velocity.get_data_in_units()
velocity_rate = wheel_velocity.rate
velocity_start = wheel_velocity.starting_time

# Reconstruct velocity timestamps
velocity_times = np.arange(len(velocity_data)) / velocity_rate + velocity_start

print("Wheel velocity:")
print(f"  Number of samples: {len(velocity_data)}")
print(f"  Sampling rate: {velocity_rate} Hz")
print(f"  Time range: {velocity_times[0]:.3f}s - {velocity_times[-1]:.3f}s")
print(f"  Unit: {wheel_velocity.unit}")
print(f"  Velocity range: {velocity_data.min():.2f} to {velocity_data.max():.2f} {wheel_velocity.unit}")

In [ ]:
# Load detected wheel movements
wheel_movements = wheel_module["WheelMovement"].to_dataframe()

print("Detected wheel movements:")
print(f"  Number of movements: {len(wheel_movements)}")
print(f"  Columns: {list(wheel_movements.columns)}")
print("\nFirst 5 movements:")
wheel_movements.head()

## Understanding Quadrature Encoding

The rotary encoder uses **quadrature encoding** with two phase-shifted channels:

```
Channel A:  --+   +---+   +---+   +--
              |   |   |   |   |   |
              +---+   +---+   +---+

Channel B:    +---+   +---+   +---+
              |   |   |   |   |   |
            --+   +---+   +---+   +--

              90 deg phase offset determines direction
```

**How direction is determined:**
- If Channel A leads Channel B: clockwise rotation
- If Channel B leads Channel A: counter-clockwise rotation

**X4 Encoding:**
- Counts all edges (rising and falling) on both channels
- Provides 4x the resolution: 1024 ticks/rev becomes 4096 effective ticks/rev
- Angular resolution: 2*pi / 4096 = 0.00153 radians (~0.088 degrees)

In [ ]:
# Simulate quadrature encoder signals to illustrate the encoding principle
# This shows how the phase relationship encodes direction

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

# Create simulated encoder signals for one direction
t = np.linspace(0, 0.1, 1000)
freq = 50  # Hz

# Channel A and B with 90-degree phase shift (counter-clockwise)
channel_a = (np.sin(2 * np.pi * freq * t) > 0).astype(int)
channel_b = (np.sin(2 * np.pi * freq * t - np.pi/2) > 0).astype(int)  # 90 deg lag

# Plot channels
axes[0].step(t * 1000, channel_a, where='post', color='blue', linewidth=1.5)
axes[0].set_ylabel('Channel A', fontsize=10)
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Low', 'High'])
axes[0].set_title('Quadrature Encoder Signals (Counter-Clockwise Rotation)', fontsize=12, fontweight='bold')

axes[1].step(t * 1000, channel_b, where='post', color='red', linewidth=1.5)
axes[1].set_ylabel('Channel B', fontsize=10)
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Low', 'High'])

# Compute cumulative position from X4 decoding
# Direction is determined by which channel leads at each edge
position = np.zeros_like(t)
ticks_per_rev = 4096  # X4 encoding
for i in range(1, len(t)):
    # Detect edges on channel A
    if channel_a[i] != channel_a[i-1]:
        # Direction based on channel B state
        if channel_a[i] == 1:  # Rising edge on A
            direction = 1 if channel_b[i] == 0 else -1
        else:  # Falling edge on A
            direction = 1 if channel_b[i] == 1 else -1
        position[i] = position[i-1] + direction * (2 * np.pi / ticks_per_rev)
    # Detect edges on channel B
    elif channel_b[i] != channel_b[i-1]:
        if channel_b[i] == 1:  # Rising edge on B
            direction = 1 if channel_a[i] == 1 else -1
        else:  # Falling edge on B
            direction = 1 if channel_a[i] == 0 else -1
        position[i] = position[i-1] + direction * (2 * np.pi / ticks_per_rev)
    else:
        position[i] = position[i-1]

axes[2].plot(t * 1000, position, color='green', linewidth=1.5)
axes[2].set_ylabel('Position (rad)', fontsize=10)
axes[2].set_xlabel('Time (ms)', fontsize=10)

plt.tight_layout()
plt.show()

print("\nX4 Encoding:")
print("  Base encoder: 1024 ticks/revolution")
print("  X4 encoding: 4096 effective ticks/revolution")
print(f"  Angular resolution: {2*np.pi/4096:.5f} rad = {360/4096:.3f} degrees")

## Visualizing Wheel Data

Let's visualize the processed wheel data to see how it looks during a typical session.

In [ ]:
# Overview of entire session
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Position
axes[0].plot(position_times, position_data, 'b-', linewidth=0.5, alpha=0.8)
axes[0].set_ylabel('Position (rad)', fontsize=10)
axes[0].set_title('Wheel Data Overview - Full Session', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Velocity
axes[1].plot(velocity_times, velocity_data, 'g-', linewidth=0.5, alpha=0.8)
axes[1].set_ylabel('Velocity (rad/s)', fontsize=10)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1].grid(True, alpha=0.3)

# Movement intervals
for _, move in wheel_movements.iterrows():
    axes[2].axvspan(move['start_time'], move['stop_time'], alpha=0.5, color='orange')
axes[2].set_ylabel('Movements', fontsize=10)
axes[2].set_xlabel('Time (s)', fontsize=10)
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

print(f"Session duration: {position_times[-1] - position_times[0]:.1f} seconds")
print(f"Total wheel rotation: {(position_data[-1] - position_data[0]) / (2*np.pi):.1f} revolutions")

In [ ]:
# Analyze wheel position in degrees across task vs passive epochs
# Convert radians to degrees
position_degrees = np.degrees(position_data)

# Get epochs from processed file
epochs_df = nwbfile_processed.epochs.to_dataframe()
print("Session epochs:")
print(f"Columns: {list(epochs_df.columns)}")
print(epochs_df)

# Find task and passive epochs
task_epoch = epochs_df[epochs_df['protocol_type'] == 'task'].iloc[0]
passive_epoch = epochs_df[epochs_df['protocol_type'] == 'passive'].iloc[0]

print(f"\nTask epoch: {task_epoch['start_time']:.1f}s - {task_epoch['stop_time']:.1f}s ({(task_epoch['stop_time'] - task_epoch['start_time'])/60:.1f} min)")
print(f"Passive epoch: {passive_epoch['start_time']:.1f}s - {passive_epoch['stop_time']:.1f}s ({(passive_epoch['stop_time'] - passive_epoch['start_time'])/60:.1f} min)")

# 2x2 comparison: Task vs Passive epochs (position in degrees)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Define time windows
task_start = task_epoch['start_time']
task_end = task_epoch['stop_time']
passive_start = passive_epoch['start_time']
passive_end = passive_epoch['stop_time']

# Short windows (2-3 minutes from start of each epoch)
task_short_start = task_start + 60  # Start 1 min into task
task_short_end = task_short_start + 180  # 3 min window
passive_short_start = passive_start + 60  # Start 1 min into passive
passive_short_end = passive_short_start + 180  # 3 min window

# Masks for each view
task_short_mask = (position_times >= task_short_start) & (position_times <= task_short_end)
passive_short_mask = (position_times >= passive_short_start) & (position_times <= passive_short_end)
task_full_mask = (position_times >= task_start) & (position_times <= task_end)
passive_full_mask = (position_times >= passive_start) & (position_times <= passive_end)

# Top-left: Task epoch short window
axes[0, 0].plot(position_times[task_short_mask] - task_short_start, 
                position_degrees[task_short_mask], 'b-', linewidth=0.8)
axes[0, 0].set_ylabel('Position (degrees)', fontsize=10)
axes[0, 0].set_xlabel('Time from window start (s)', fontsize=10)
axes[0, 0].set_title(f'Task Epoch - 3 min window\n(t={task_short_start:.0f}s to {task_short_end:.0f}s)', 
                     fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Top-right: Passive epoch short window
axes[0, 1].plot(position_times[passive_short_mask] - passive_short_start, 
                position_degrees[passive_short_mask], 'r-', linewidth=0.8)
axes[0, 1].set_ylabel('Position (degrees)', fontsize=10)
axes[0, 1].set_xlabel('Time from window start (s)', fontsize=10)
axes[0, 1].set_title(f'Passive Epoch - 3 min window\n(t={passive_short_start:.0f}s to {passive_short_end:.0f}s)', 
                     fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Bottom-left: Full task epoch
axes[1, 0].plot(position_times[task_full_mask] - task_start, 
                position_degrees[task_full_mask], 'b-', linewidth=0.5)
axes[1, 0].set_ylabel('Position (degrees)', fontsize=10)
axes[1, 0].set_xlabel('Time from epoch start (s)', fontsize=10)
axes[1, 0].set_title(f'Task Epoch - Full ({(task_end - task_start)/60:.1f} min)', 
                     fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Bottom-right: Full passive epoch
axes[1, 1].plot(position_times[passive_full_mask] - passive_start, 
                position_degrees[passive_full_mask], 'r-', linewidth=0.5)
axes[1, 1].set_ylabel('Position (degrees)', fontsize=10)
axes[1, 1].set_xlabel('Time from epoch start (s)', fontsize=10)
axes[1, 1].set_title(f'Passive Epoch - Full ({(passive_end - passive_start)/60:.1f} min)', 
                     fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Wheel Position: Task vs Passive Epochs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistics
task_pos = position_degrees[task_full_mask]
passive_pos = position_degrees[passive_full_mask]

print("\nPosition statistics:")
print("Task epoch:")
print(f"  Start position: {task_pos[0]:.1f} deg")
print(f"  End position: {task_pos[-1]:.1f} deg")
print(f"  Net change: {task_pos[-1] - task_pos[0]:.1f} deg ({(task_pos[-1] - task_pos[0])/360:.1f} revolutions)")
print(f"  Range: {task_pos.min():.1f} to {task_pos.max():.1f} deg")

print("\nPassive epoch:")
print(f"  Start position: {passive_pos[0]:.1f} deg")
print(f"  End position: {passive_pos[-1]:.1f} deg")
print(f"  Net change: {passive_pos[-1] - passive_pos[0]:.1f} deg ({(passive_pos[-1] - passive_pos[0])/360:.1f} revolutions)")
print(f"  Range: {passive_pos.min():.1f} to {passive_pos.max():.1f} deg")

# Calculate drift rate in passive
passive_duration = (passive_end - passive_start) / 60  # minutes
passive_drift = passive_pos[-1] - passive_pos[0]
drift_rate = passive_drift / passive_duration

print("\nPassive drift analysis:")
print(f"  Duration: {passive_duration:.1f} minutes")
print(f"  Total drift: {passive_drift:.1f} degrees")
print(f"  Drift rate: {drift_rate:.1f} deg/min ({drift_rate/360:.2f} rev/min)")

In [ ]:
# Analyze velocity during task vs passive epochs
# Convert velocity to degrees/s for consistency
velocity_degrees = np.degrees(velocity_data)

# 2x2 comparison: Task vs Passive velocity
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Masks for velocity data (using velocity_times)
task_short_vel_mask = (velocity_times >= task_short_start) & (velocity_times <= task_short_end)
passive_short_vel_mask = (velocity_times >= passive_short_start) & (velocity_times <= passive_short_end)
task_full_vel_mask = (velocity_times >= task_start) & (velocity_times <= task_end)
passive_full_vel_mask = (velocity_times >= passive_start) & (velocity_times <= passive_end)

# Top-left: Task epoch short window velocity
axes[0, 0].plot(velocity_times[task_short_vel_mask] - task_short_start, 
                velocity_degrees[task_short_vel_mask], 'b-', linewidth=0.5)
axes[0, 0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0, 0].set_ylabel('Velocity (deg/s)', fontsize=10)
axes[0, 0].set_xlabel('Time from window start (s)', fontsize=10)
axes[0, 0].set_title('Task Epoch - 3 min window (Velocity)', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Top-right: Passive epoch short window velocity
axes[0, 1].plot(velocity_times[passive_short_vel_mask] - passive_short_start, 
                velocity_degrees[passive_short_vel_mask], 'r-', linewidth=0.5)
axes[0, 1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[0, 1].set_ylabel('Velocity (deg/s)', fontsize=10)
axes[0, 1].set_xlabel('Time from window start (s)', fontsize=10)
axes[0, 1].set_title('Passive Epoch - 3 min window (Velocity)', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Bottom-left: Full task epoch velocity
axes[1, 0].plot(velocity_times[task_full_vel_mask] - task_start, 
                velocity_degrees[task_full_vel_mask], 'b-', linewidth=0.3, alpha=0.7)
axes[1, 0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1, 0].set_ylabel('Velocity (deg/s)', fontsize=10)
axes[1, 0].set_xlabel('Time from epoch start (s)', fontsize=10)
axes[1, 0].set_title('Task Epoch - Full (Velocity)', fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Bottom-right: Full passive epoch velocity
axes[1, 1].plot(velocity_times[passive_full_vel_mask] - passive_start, 
                velocity_degrees[passive_full_vel_mask], 'r-', linewidth=0.3, alpha=0.7)
axes[1, 1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1, 1].set_ylabel('Velocity (deg/s)', fontsize=10)
axes[1, 1].set_xlabel('Time from epoch start (s)', fontsize=10)
axes[1, 1].set_title('Passive Epoch - Full (Velocity)', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Wheel Velocity: Task vs Passive Epochs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Velocity statistics
task_vel = velocity_degrees[task_full_vel_mask]
passive_vel = velocity_degrees[passive_full_vel_mask]

print("Velocity statistics:")
print("Task epoch:")
print(f"  Mean velocity: {task_vel.mean():.2f} deg/s")
print(f"  Std velocity: {task_vel.std():.2f} deg/s")
print(f"  Range: {task_vel.min():.1f} to {task_vel.max():.1f} deg/s")
print(f"  % time moving (|vel| > 10 deg/s): {(np.abs(task_vel) > 10).mean() * 100:.1f}%")

print("\nPassive epoch:")
print(f"  Mean velocity: {passive_vel.mean():.2f} deg/s")
print(f"  Std velocity: {passive_vel.std():.2f} deg/s")
print(f"  Range: {passive_vel.min():.1f} to {passive_vel.max():.1f} deg/s")
print(f"  % time moving (|vel| > 10 deg/s): {(np.abs(passive_vel) > 10).mean() * 100:.1f}%")

# Check if velocity is biased in one direction during passive
print("\nVelocity bias analysis:")
print(f"  Task - positive velocity %: {(task_vel > 0).mean() * 100:.1f}%")
print(f"  Passive - positive velocity %: {(passive_vel > 0).mean() * 100:.1f}%")

In [ ]:
# Deeper analysis of passive wheel behavior
# Is this continuous drift or discrete movements?

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Velocity histogram comparison
ax = axes[0, 0]
ax.hist(task_vel, bins=100, alpha=0.5, label='Task', color='blue', density=True)
ax.hist(passive_vel, bins=100, alpha=0.5, label='Passive', color='red', density=True)
ax.set_xlabel('Velocity (deg/s)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Velocity Distribution: Task vs Passive', fontsize=11, fontweight='bold')
ax.legend()
ax.set_xlim(-600, 600)

# 2. Zoom in on small velocities to see if there's constant drift
ax = axes[0, 1]
ax.hist(task_vel, bins=200, alpha=0.5, label='Task', color='blue', density=True, range=(-50, 50))
ax.hist(passive_vel, bins=200, alpha=0.5, label='Passive', color='red', density=True, range=(-50, 50))
ax.axvline(0, color='black', linestyle='--', linewidth=0.5)
ax.axvline(passive_vel.mean(), color='red', linestyle='-', linewidth=2, label=f'Passive mean: {passive_vel.mean():.1f} deg/s')
ax.axvline(task_vel.mean(), color='blue', linestyle='-', linewidth=2, label=f'Task mean: {task_vel.mean():.1f} deg/s')
ax.set_xlabel('Velocity (deg/s)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Velocity Distribution (Zoomed)', fontsize=11, fontweight='bold')
ax.legend()

# 3. Check detected movements during passive
passive_moves = wheel_movements[(wheel_movements['start_time'] >= passive_start) & 
                                 (wheel_movements['stop_time'] <= passive_end)]
task_moves = wheel_movements[(wheel_movements['start_time'] >= task_start) & 
                              (wheel_movements['stop_time'] <= task_end)]

ax = axes[1, 0]
ax.bar(['Task', 'Passive'], [len(task_moves), len(passive_moves)], color=['blue', 'red'], alpha=0.7)
ax.set_ylabel('Number of detected movements', fontsize=10)
ax.set_title('Detected Wheel Movements per Epoch', fontsize=11, fontweight='bold')

# Add text with movement rates
task_rate = len(task_moves) / ((task_end - task_start) / 60)
passive_rate = len(passive_moves) / ((passive_end - passive_start) / 60)
ax.text(0, len(task_moves) + 10, f'{task_rate:.1f}/min', ha='center', fontsize=10)
ax.text(1, len(passive_moves) + 10, f'{passive_rate:.1f}/min', ha='center', fontsize=10)

# 4. Movement amplitude comparison
ax = axes[1, 1]
if len(task_moves) > 0 and len(passive_moves) > 0:
    ax.hist(np.abs(np.degrees(task_moves['peak_amplitude'])), bins=50, alpha=0.5, 
            label=f'Task (n={len(task_moves)})', color='blue', density=True)
    ax.hist(np.abs(np.degrees(passive_moves['peak_amplitude'])), bins=50, alpha=0.5, 
            label=f'Passive (n={len(passive_moves)})', color='red', density=True)
    ax.set_xlabel('Movement Amplitude (degrees)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title('Movement Amplitude Distribution', fontsize=11, fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 500)

plt.suptitle('Passive Wheel Behavior Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics
print("=" * 60)
print("PASSIVE WHEEL BEHAVIOR ANALYSIS")
print("=" * 60)

print("\nDetected movements:")
print(f"  Task: {len(task_moves)} movements ({task_rate:.1f}/min)")
print(f"  Passive: {len(passive_moves)} movements ({passive_rate:.1f}/min)")

print("\nMean velocity (should be ~0 for stationary wheel):")
print(f"  Task: {task_vel.mean():.2f} deg/s")
print(f"  Passive: {passive_vel.mean():.2f} deg/s  <-- BIASED!")

print("\nTime spent at non-zero velocity (|vel| > 1 deg/s):")
print(f"  Task: {(np.abs(task_vel) > 1).mean() * 100:.1f}%")
print(f"  Passive: {(np.abs(passive_vel) > 1).mean() * 100:.1f}%")

# Check if passive has consistent positive velocity (drift in one direction)
print("\nVelocity direction bias:")
print(f"  Task positive velocity: {(task_vel > 1).mean() * 100:.1f}%")
print(f"  Passive positive velocity: {(passive_vel > 1).mean() * 100:.1f}%  <-- STRONGLY BIASED!")

print("\n" + "=" * 60)
print("INTERPRETATION:")
print("=" * 60)
print("""
The passive epoch shows:
1. A strongly positive mean velocity (wheel consistently turning one direction)
2. Many detected movements, similar rate to task
3. This is NOT encoder drift (would be random, not directional)

LIKELY EXPLANATION: 
During passive replay, the mouse is not engaged in the task but the wheel 
remains FREE (not locked). The mouse may be:
- Fidgeting/grooming with paws touching wheel
- Bored and spinning wheel habitually
- Making spontaneous movements

This is EXPECTED BEHAVIOR - the IBL passive protocol does not lock the wheel.
The neural responses during passive can still be analyzed, but wheel position
during passive should be interpreted carefully.
""")

In [ ]:
# Detailed view of a few seconds showing individual movements
# Find a region with some movements
start_idx = 0
for i, move in wheel_movements.iterrows():
    if move['start_time'] > 100:  # Start after 100 seconds
        start_idx = i
        break

# Get time window around some movements
t_start = wheel_movements.iloc[start_idx]['start_time'] - 1
t_end = t_start + 10  # 10 second window

# Mask for position data
pos_mask = (position_times >= t_start) & (position_times <= t_end)
vel_mask = (velocity_times >= t_start) & (velocity_times <= t_end)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Position with event-driven sampling visible
axes[0].plot(position_times[pos_mask], position_data[pos_mask], 'b.-', 
             linewidth=0.5, markersize=2, alpha=0.8)
axes[0].set_ylabel('Position (rad)', fontsize=10)
axes[0].set_title(f'Wheel Data Detail - {t_start:.1f}s to {t_end:.1f}s', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Velocity with movement intervals
axes[1].plot(velocity_times[vel_mask], velocity_data[vel_mask], 'g-', linewidth=1)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.5)

# Highlight detected movements
for _, move in wheel_movements.iterrows():
    if move['start_time'] >= t_start and move['stop_time'] <= t_end:
        axes[1].axvspan(move['start_time'], move['stop_time'], alpha=0.3, color='orange', 
                       label='Detected movement' if _ == start_idx else '')

axes[1].set_ylabel('Velocity (rad/s)', fontsize=10)
axes[1].set_xlabel('Time (s)', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

# Show sampling characteristics
dt = np.diff(position_times[pos_mask])
print("\nPosition sampling characteristics:")
print(f"  Mean inter-sample interval: {np.mean(dt)*1000:.2f} ms")
print(f"  Min inter-sample interval: {np.min(dt)*1000:.2f} ms")
print(f"  Max inter-sample interval: {np.max(dt)*1000:.2f} ms")
print("  Note: Variable sampling because encoder only fires on wheel movement")

## Movement Detection Analysis

The IBL pipeline detects discrete movements from the continuous position data using these criteria:
- Minimum displacement: ~0.012 radians (~8 encoder ticks) over 200ms
- Minimum duration: 50ms
- Merge threshold: Movements within 100ms are combined

In [ ]:
# Analyze movement statistics
wheel_movements['duration'] = wheel_movements['stop_time'] - wheel_movements['start_time']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Movement duration histogram
axes[0].hist(wheel_movements['duration'] * 1000, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Movement Duration (ms)', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].set_title('Movement Duration Distribution', fontsize=11, fontweight='bold')
axes[0].axvline(wheel_movements['duration'].median() * 1000, color='red', linestyle='--', 
               label=f'Median: {wheel_movements["duration"].median()*1000:.0f}ms')
axes[0].legend()

# Peak amplitude histogram
axes[1].hist(np.abs(wheel_movements['peak_amplitude']), bins=50, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Peak Amplitude (rad)', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('Movement Amplitude Distribution', fontsize=11, fontweight='bold')

# Inter-movement interval
imi = np.diff(wheel_movements['start_time'].values)
axes[2].hist(imi, bins=50, color='seagreen', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Inter-Movement Interval (s)', fontsize=10)
axes[2].set_ylabel('Count', fontsize=10)
axes[2].set_title('Time Between Movements', fontsize=11, fontweight='bold')
axes[2].set_xlim(0, 5)

plt.tight_layout()
plt.show()

print("Movement statistics:")
print(f"  Total movements: {len(wheel_movements)}")
print(f"  Median duration: {wheel_movements['duration'].median()*1000:.0f} ms")
print(f"  Median amplitude: {np.abs(wheel_movements['peak_amplitude']).median():.3f} rad")
print(f"  Median inter-movement interval: {np.median(imi):.2f} s")

## Signal Flow Summary

The wheel data flows through the IBL pipeline as follows:

```
1. HARDWARE: Rotary encoder (1024 ticks/rev)
       |
       v
2. NIDQ: Records TTL pulses on channels 5 & 6 (~1 kHz)
       |
       v
3. PREPROCESSING: X4 decoding converts pulses to position
       |
       v
4. ALF FILES: wheel.position.npy, wheel.timestamps.npy
       |
       v
5. MOVEMENT DETECTION: wheelMoves.intervals.npy
       |
       v
6. NWB CONVERSION: WheelPosition, WheelSmoothedVelocity, WheelMovement, etc.
```

In [ ]:
# Create a summary visualization showing the data processing pipeline
fig, ax = plt.subplots(figsize=(14, 8))

# Draw pipeline boxes
boxes = [
    (0.1, 0.85, 'HARDWARE\nRotary Encoder\n(1024 ticks/rev)', 'lightblue'),
    (0.1, 0.65, 'NIDQ DAQ\nChannels 5 & 6\n(~1 kHz sampling)', 'lightgreen'),
    (0.1, 0.45, 'IBL Preprocessing\nX4 Quadrature Decoding\n(4096 effective ticks)', 'lightyellow'),
    (0.1, 0.25, 'ALF Files\nwheel.position.npy\nwheel.timestamps.npy', 'lightcoral'),
    (0.1, 0.05, 'NWB File\nWheelPosition\nWheelSmoothedVelocity', 'plum'),
]

for x, y, text, color in boxes:
    rect = plt.Rectangle((x, y), 0.35, 0.15, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 0.175, y + 0.075, text, ha='center', va='center', fontsize=10, fontweight='bold')

# Draw arrows
for i in range(len(boxes) - 1):
    ax.annotate('', xy=(0.275, boxes[i+1][1] + 0.15), xytext=(0.275, boxes[i][1]),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Add data examples on the right
examples = [
    (0.55, 0.85, 'Two phase-shifted\nTTL signals (A & B)'),
    (0.55, 0.65, 'Digital pulses at\nP0.5 and P0.6'),
    (0.55, 0.45, 'Position: radians\nResolution: 0.00153 rad'),
    (0.55, 0.25, f'Samples: {len(position_data):,}\nTime: {position_times[-1]:.0f}s'),
    (0.55, 0.05, '+ Velocity @ 1000 Hz\n+ Movement intervals'),
]

for x, y, text in examples:
    ax.text(x, y + 0.075, text, ha='left', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Wheel Data Processing Pipeline', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
# Close NWB file handles when done
# io_raw.close()
# io_processed.close()